# Does learning matter? Trained vs untrained AlexNet
**Reading:** Cichy, R. M., Khosla, A., Pantazis, D., Torralba, A., & Oliva, A. (2016). Comparison of deep neural networks to spatio-temporal cortical dynamics of human visual object recognition reveals hierarchical correspondence. *Scientific Reports, 6*, 27755.

Cichy et al. compared AlexNet-like networks with human brain responses. As a control, they also tested an **untrained** network: the same architecture, but with random weights. Today we do the same: we compare a **trained** and an **untrained** AlexNet.

**What you will do**
1. Load a trained and an untrained AlexNet
2. Classify two photos with both networks
3. Pick an image and find its **nearest neighbours** in every layer: which of 120 photos of animals, fruit and objects does each network find most similar?
4. Count across all images: from which layer on are the neighbours the **same concept** (lion – lion) or the **same kind** (lion – koala)?
5. See all pairs of images at once: **similarity matrices** and a **map** of all 120 images


## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

In [ ]:
# Find the course data in Sciebo (works even if the shared folder was renamed)
matches = list(Path.home().glob("sciebo/*/Week4/"))
if not matches:
    raise FileNotFoundError(
        "Course data not found. Did you connect Sciebo when starting the server, "
        "and accept the shared CNU folder in Sciebo?"
    )
DATA = matches[0]
IMAGE_DIR = DATA / "animals_objects"              # 120 photos for today: animals, fruit and objects
WEEK3_DIR = DATA.parent / "Week3" / "images"      # we reuse two photos from Week 3
print(f"Using data from {DATA}")

## 1. Load a trained and an untrained AlexNet
Both networks have exactly the same **architecture** (the same layers). The only difference is the **weights**:
- the **trained** network learned its weights from 1.2 million ImageNet photos,
- the **untrained** network has **random** weights, as before any training.

In [ ]:
weights = models.AlexNet_Weights.IMAGENET1K_V1
categories = weights.meta["categories"]

# Trained: the ImageNet weights (as in Week 2 and 3)
trained = models.alexnet(weights=weights).to(device).eval()

# Untrained: same architecture, random weights.
# torch.manual_seed fixes the random numbers, so everyone gets the same "random" network.
torch.manual_seed(0)
untrained = models.alexnet(weights=None).to(device).eval()

networks = {"trained": trained, "untrained": untrained}

## 2. How images are prepared (as in Week 2 and 3)

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

def classify(model, imgs, k=5):
    """Return the k most likely labels and their probabilities for each image."""
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(batch), dim=1)
    top_probs, top_idx = probs.topk(k, dim=1)
    return [[(categories[i], p.item()) for p, i in zip(p_row, i_row)]
            for p_row, i_row in zip(top_probs, top_idx)]

def show_top5(images, names, results, title=None):
    """One row per image: the image on the left, its top-5 predictions on the right."""
    n = len(images)
    fig, axes = plt.subplots(n, 2, figsize=(9, 2.3 * n), gridspec_kw={"width_ratios": [1, 1.7]})
    axes = np.atleast_2d(axes)
    for row, (img, name, res) in enumerate(zip(images, names, results)):
        ax_img, ax_bar = axes[row]
        ax_img.imshow(what_network_sees(img)); ax_img.axis("off")
        ax_img.set_title(name, fontsize=10)
        labels = [label for label, _ in res][::-1]
        probs  = [p for _, p in res][::-1]
        ax_bar.barh(labels, probs, color="tab:blue")
        ax_bar.set_xlim(0, 1)
        for y, p in enumerate(probs):
            ax_bar.text(p + 0.01, y, f"{p:.2f}", va="center", fontsize=9)
    axes[-1, 1].set_xlabel("probability")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

## 3. Classify two photos with both networks
These are the zebra and plane photos from Week 3. The trained network recognised both with high confidence.

**Before running:** what do you expect from the untrained network?

In [ ]:
photo_names = ["zebra_photo", "plane_photo"]
photos = [Image.open(next(WEEK3_DIR.glob(name + ".*"))).convert("RGB") for name in photo_names]

for net_name, model in networks.items():
    show_top5(photos, photo_names, classify(model, photos), title=f"{net_name} AlexNet")

**Think about it:**
- What does the untrained network answer? How confident is it?
- Both networks have the same layers. So what exactly did the trained network learn?

## 4. Which images does a network find similar?
So far we looked at single images. Now we ask: **which images does a network treat as similar?**

For each image, we take the activations of a layer: one long list of numbers. Two images are **similar** for the network if their activation patterns are similar. We measure the **dissimilarity** as `1 − correlation`:
- **0** = identical activation patterns,
- **1** = unrelated patterns,
- up to **2** = opposite patterns.

This is the measure Cichy et al. used to compare networks with the brain.

Our images have three levels:

| level | example |
|---|---|
| **kind** | animal, fruit or object |
| **concept** | lion, banana, violin, ... (4 per kind) |
| **photo** | 10 different photos of each concept |

That makes 3 × 4 × 10 = 120 images.

In [ ]:
# File names: <kind>_<concept>_<number>.jpg, e.g. animal_lion_1.jpg
KINDS = ["animal", "fruit", "object"]

paths = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
               and not p.name.startswith("."))
info = [p.stem.split("_") for p in paths]       # [kind, concept, number]
order = sorted(range(len(paths)), key=lambda i: (KINDS.index(info[i][0]), info[i][1], int(info[i][2])))
paths = [paths[i] for i in order]
kinds = np.array([info[i][0] for i in order])
concepts = np.array([info[i][1] for i in order])
names = [p.stem for p in paths]
images = [Image.open(p).convert("RGB") for p in paths]
concept_list = [str(c) for c in dict.fromkeys(concepts)]    # unique, in order
print(f"{len(images)} images, {len(concept_list)} concepts: {concept_list}")

# One row per concept; the image names are what you use as TARGET below
n_per = max(int(i[2]) for i in info)
fig, axes = plt.subplots(len(concept_list), n_per, figsize=(1.6 * n_per, 1.75 * len(concept_list)))
for ax in axes.ravel():
    ax.axis("off")
for r, c in enumerate(concept_list):
    for k, i in enumerate(np.where(concepts == c)[0]):
        axes[r, k].imshow(what_network_sees(images[i]))
        axes[r, k].set_title(names[i], fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# All layers of AlexNet: the ReLU after each convolution and fully connected layer, and the output (fc8)
LAYERS = {
    "conv1": ("features", 1),
    "conv2": ("features", 4),
    "conv3": ("features", 7),
    "conv4": ("features", 9),
    "conv5": ("features", 11),
    "fc6":   ("classifier", 2),
    "fc7":   ("classifier", 5),
    "fc8":   ("classifier", 6),     # the 1,000 outputs, before the softmax
}

def get_activations(model, imgs):
    # Return a dict: layer name -> array of shape (number of images, number of units).
    saved = {}
    def make_hook(name):
        def hook(layer, inputs, output):
            out = output.detach()
            saved[name] = out.reshape(out.shape[0], -1).numpy()   # one long vector per image
        return hook
    hooks = [getattr(model, part)[i].register_forward_hook(make_hook(name))
             for name, (part, i) in LAYERS.items()]
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)
    with torch.no_grad():
        model(batch)
    for h in hooks:
        h.remove()
    return saved

def dissimilarity(target, others):
    # 1 - correlation between one activation pattern and each row of `others`.
    t = (target - target.mean()) / (target.std() + 1e-12)
    o = (others - others.mean(axis=1, keepdims=True)) / (others.std(axis=1, keepdims=True) + 1e-12)
    return 1 - (o @ t) / len(t)

# Activations of all 120 images, for both networks (takes a few seconds)
acts = {net_name: get_activations(model, images) for net_name, model in networks.items()}

## 5. Nearest neighbours of one image
Pick a **target** image: for every layer, the cells below show the **5 images that are most similar** to it (lowest dissimilarity, shown in brackets).

- The title of each neighbour is coloured **green** if it is the same kind as the target (animal, fruit or object) and **red** if it is a different kind.
- With `EXCLUDE_SAME_CONCEPT = True`, the other photos of the target's own concept are left out. Then the question becomes: *apart from other lions, what does the network find similar to this lion?*

### Your turn first
**Before running the next cell**, predict for the **trained** and the **untrained** network:
- In which layers will the other lions be the nearest neighbours of a lion?
- In the early layers, what will the neighbours have in common with the lion?

In [ ]:
def show_neighbours(target_acts, target_img, target_name, target_kind, net_name, exclude=(), k=5):
    # One row per layer: the target and its k most similar images.
    fig, axes = plt.subplots(len(LAYERS), k + 1, figsize=(1.7 * (k + 1), 1.9 * len(LAYERS)))
    for r, layer in enumerate(LAYERS):
        d = dissimilarity(target_acts[layer], acts[net_name][layer])
        d[list(exclude)] = np.inf                        # never pick these
        best = np.argsort(d)[:k]
        axes[r, 0].imshow(what_network_sees(target_img))
        axes[r, 0].set_title(target_name if r == 0 else "", fontsize=8)
        axes[r, 0].set_ylabel(layer, fontsize=10)
        for c, i in enumerate(best, start=1):
            axes[r, c].imshow(what_network_sees(images[i]))
            color = "tab:green" if kinds[i] == target_kind else "tab:red"
            axes[r, c].set_title(f"{concepts[i]} ({d[i]:.2f})", fontsize=8, color=color)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{net_name} AlexNet: nearest neighbours of {target_name}", y=1.0)
    plt.tight_layout(); plt.show()

In [ ]:
TARGET = "animal_lion_1"          # any image name from section 4, e.g. "fruit_banana_3"
EXCLUDE_SAME_CONCEPT = False       # True: leave out the other photos of the same concept

i = names.index(TARGET)
exclude = {i}                                            # the target itself
if EXCLUDE_SAME_CONCEPT:
    exclude |= set(np.where(concepts == concepts[i])[0])
for net_name in networks:
    target_acts = {layer: acts[net_name][layer][i] for layer in LAYERS}
    show_neighbours(target_acts, images[i], TARGET, kinds[i], net_name, exclude)

**Think about it:**
- From which layer on are the other lions the nearest neighbours? Is it the same in the untrained network?
- In the early layers, what do the neighbours have in common with the target: colour, texture, background, shape?
- Set `EXCLUDE_SAME_CONCEPT = True`. Are the neighbours in `fc7` and `fc8` mostly **green** (same kind)? And in the untrained network?
- Try other targets: a fruit, an object. Is it the same for all of them? Which images are "hard" for the network?

### Try it: your own image
Upload a photo (e.g. of your pet, your breakfast or your bike) into the file browser on the left and set its file name below. Which of our 120 images does each layer find most similar?

In [ ]:
MY_IMAGE = "my_photo.jpg"          # the file you uploaded
MY_KIND = "animal"                 # "animal", "fruit" or "object": used for the green/red colours

if not Path(MY_IMAGE).exists():
    print(f"Upload a photo and set MY_IMAGE to its file name ({MY_IMAGE!r} not found).")
else:
    my_img = Image.open(MY_IMAGE).convert("RGB")
    for net_name, model in networks.items():
        my_acts = {layer: a[0] for layer, a in get_activations(model, [my_img]).items()}
        show_neighbours(my_acts, my_img, MY_IMAGE, MY_KIND, net_name)

## 6. Counting across all images
One target can be a lucky (or unlucky) case. So now we use **every** image as the target once and count its 5 nearest neighbours:

- **same concept:** how many of the 5 are photos of the target's own concept (lion – lion)?
- **same kind:** with the target's own concept excluded, how many of the 5 are the same kind (lion – koala)?

We plot the percentage for every layer. The dotted lines show **chance**: what you would get if the neighbours were picked at random (same concept: 9 of the other 119 images; same kind: 30 of the remaining 110).

In [ ]:
def neighbour_counts(net_name, layer, k=5):
    # Percentage of the k nearest neighbours that are the same concept / the same kind.
    A = acts[net_name][layer]
    same_concept, same_kind = [], []
    for i in range(len(images)):
        d = dissimilarity(A[i], A)
        d[i] = np.inf                                         # not the target itself
        best = np.argsort(d)[:k]
        same_concept.append(np.mean(concepts[best] == concepts[i]))
        d[concepts == concepts[i]] = np.inf                   # now also without its own concept
        best = np.argsort(d)[:k]
        same_kind.append(np.mean(kinds[best] == kinds[i]))
    return 100 * np.mean(same_concept), 100 * np.mean(same_kind)

layers = list(LAYERS)
n = len(images)
n_concept = (concepts == concepts[0]).sum()                  # photos per concept
n_kind = (kinds == kinds[0]).sum()                           # images per kind
chance = [100 * (n_concept - 1) / (n - 1), 100 * (n_kind - n_concept) / (n - n_concept)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for net_name, style in [("trained", "o-"), ("untrained", "o--")]:
    values = np.array([neighbour_counts(net_name, layer) for layer in layers])
    axes[0].plot(layers, values[:, 0], style, label=net_name)
    axes[1].plot(layers, values[:, 1], style, label=net_name)
for ax, ch, title in zip(axes, chance, ["same concept (lion – lion)",
                                        "same kind, own concept excluded (lion – koala)"]):
    ax.axhline(ch, color="gray", ls=":", label="chance")
    ax.set_ylim(0, 100)
    ax.set_title(title); ax.set_xlabel("layer")
    ax.legend()
axes[0].set_ylabel("% of the 5 nearest neighbours")
plt.tight_layout(); plt.show()

**Think about it:**
- How does the "same concept" curve change from `conv1` to `fc7` in the trained network? And in the untrained network?
- The untrained network is above chance even though it learned nothing. Look at the photos: why could 10 photos of oranges be similar for *any* network?
- The "same kind" curve is much lower than the "same concept" curve. In which layer does it rise? AlexNet was trained to tell apart 1,000 labels such as "lion" and "banana", but it was never told what an "animal" is. How could that explain the result?
- Cichy et al. found that the trained network matched the brain better than the untrained one, especially for the **later** brain responses. Does the difference between your two networks also appear mainly in the later layers?

## 7. Similarity matrices
Now we look at **all pairs** of images at once. For every pair, we compute the dissimilarity (1 − correlation) of their activation patterns, and put all of them in a table: one row and one column per image. This table is called a **representational dissimilarity matrix (RDM)**. It is exactly the tool Cichy et al. used to compare networks with the brain: they computed the same kind of matrix from brain activity and compared the two.

We show it as a picture: **dark = similar, bright = dissimilar**.

In [ ]:
def rdm(A):
    # Dissimilarity (1 - correlation) between every pair of images (rows of A)
    return 1 - np.corrcoef(A)

def show_rdms(rdm_layers):
    # One matrix per network (rows) and layer (columns). White lines separate the kinds.
    centres = [np.where(concepts == c)[0].mean() for c in concept_list]
    bounds = [i - 0.5 for i in range(1, len(kinds)) if kinds[i] != kinds[i - 1]]
    fig, axes = plt.subplots(len(networks), len(rdm_layers), figsize=(4.8 * len(rdm_layers), 4.6 * len(networks)),
                             squeeze=False)
    for r, net_name in enumerate(networks):
        for c, layer in enumerate(rdm_layers):
            ax = axes[r, c]
            m = rdm(acts[net_name][layer])
            off_diag = m[~np.eye(len(m), dtype=bool)]           # all pairs of different images
            # Colour range from the pairs of different images, so small differences are visible
            im = ax.imshow(m, cmap="viridis", vmin=off_diag.min(), vmax=off_diag.max())
            for b in bounds:
                ax.axhline(b, color="white", lw=1.5); ax.axvline(b, color="white", lw=1.5)
            ax.set_xticks(centres); ax.set_xticklabels(concept_list, rotation=90, fontsize=8)
            ax.set_yticks(centres); ax.set_yticklabels(concept_list, fontsize=8)
            ax.set_title(f"{net_name}: {layer}")
            fig.colorbar(im, ax=ax, fraction=0.046, label="dissimilarity")
    fig.suptitle("Dark = similar, bright = dissimilar")
    plt.tight_layout(); plt.show()

show_rdms(["conv1", "conv5", "fc7"])

**How to read the matrices:** each square is one pair of images. The diagonal is always 0 (an image compared with itself). Each matrix has its own colour scale (see the colour bar), so compare the **patterns**, not the colours, between matrices. The images are sorted by kind, then by concept:
- if a layer groups the photos of each **concept**, you see small dark **10 × 10 blocks** along the diagonal,
- if it groups the **kinds**, you see larger dark **40 × 40 blocks** between the white lines.

**Think about it:**
- Where do you see the small concept blocks? In which network, and from which layer on? Does that fit your counts from section 6?
- Do you see the large kind blocks anywhere?
- The untrained network also shows a few dark squares. Look at those images: what do they have in common?

**Try it:** show other layers, e.g. `show_rdms(["conv3", "fc6", "fc8"])`.

## 8. A map of all 120 images
The matrices contain all the information, but a weak pattern can be hard to spot in 14,400 coloured squares. So we draw the same dissimilarities as a map: every image is a dot, and images that a layer finds **similar** end up **close together**.

The map is made with **multidimensional scaling (MDS)**.

**What is MDS?** Imagine you only had a table with the travel distances between every pair of German cities, but no map. MDS is a method that draws the map from that table: it places every city as a dot so that the distances between the dots match the distances in the table as well as possible. Here, the "cities" are our 120 images, and the "travel distances" are the dissimilarities (1 − correlation) between their activation patterns in one layer.

Three things to keep in mind:
- The axes have **no meaning** of their own (no "x = colour", no "y = size"). The map can also come out rotated or mirrored. Only the **distances between the dots** count.
- A layer has thousands of units, so the real dissimilarities cannot all be shown exactly in 2D. The map shows the **largest** differences between the images and leaves out smaller ones, so it is a simplification.
- Each layer and network gets its own map, so compare **which dots are close together**, not where they are on the page.

Colours show the **kind**, symbols show the **concept**.

In [ ]:
def dissimilarity_all(A):
    # 1 - correlation between every pair of rows (images) of A
    return 1 - np.corrcoef(A)

def mds_2d(D):
    # Classical multidimensional scaling (MDS): turn a table of dissimilarities into a 2D map.
    # Like drawing a map of cities when you only know the distances between them.
    n = len(D)
    # 1. Turn the dissimilarities into "how much do two images go together", relative to the average image
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J
    # 2. Find the two directions along which the images differ most (the largest eigenvalues of B) ...
    values, vectors = np.linalg.eigh(B)
    top = np.argsort(values)[::-1][:2]
    # 3. ... and use them as the x and y coordinates of the map
    return vectors[:, top] * np.sqrt(np.maximum(values[top], 0))

def show_maps(map_layers):
    kind_colors = dict(zip(KINDS, ["tab:orange", "tab:green", "tab:blue", "tab:purple"]))
    symbols = ["o", "s", "^", "D", "v", "P"]
    fig, axes = plt.subplots(len(networks), len(map_layers), figsize=(4.2 * len(map_layers), 4.2 * len(networks)),
                             squeeze=False)
    for r, net_name in enumerate(networks):
        for c, layer in enumerate(map_layers):
            ax = axes[r, c]
            xy = mds_2d(dissimilarity_all(acts[net_name][layer]))
            for kind in KINDS:
                kind_concepts = [con for con in concept_list if kinds[concepts == con][0] == kind]
                for s, con in enumerate(kind_concepts):
                    sel = concepts == con
                    ax.scatter(xy[sel, 0], xy[sel, 1], color=kind_colors[kind], marker=symbols[s % len(symbols)],
                               s=40, label=f"{con} ({kind})" if r == 0 and c == 0 else None)
            ax.set_title(f"{net_name}: {layer}")
            ax.set_xticks([]); ax.set_yticks([])
    fig.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

show_maps(["conv1", "conv5", "fc7", "fc8"])

**Think about it:**
- In `conv1`, are the photos of the same concept close together? What seems to decide where an image ends up (colour, brightness, background)?
- In the trained `fc7` and `fc8`: do the photos of each concept form small groups? Do the **colours** (animals, fruit, objects) form larger regions?
- Compare with the untrained network: which groups are missing there?
- In section 6, the "same kind" counts were much lower than the "same concept" counts. Does the map fit with that?
- Can you see the kind regions here more easily than the large blocks in the matrices of section 7? Why could that be?

**Try it:** show other layers, e.g. `show_maps(["conv2", "conv3", "conv4", "fc6"])`.

## 9. Discussion questions
1. The untrained network has the right architecture but has learned nothing. What can architecture alone explain (think of the photos of oranges), and what needs learning?
2. AlexNet finds photos of the **same concept** similar much more reliably than photos of the **same kind**. What does that tell you about what the network learned from its 1,000 ImageNet labels?
3. In the human brain, the inferior temporal (IT) cortex clearly separates animate from inanimate objects (Kriegeskorte et al., 2008). If you could make the matrix and the map of sections 7 and 8 from brain activity, what would you expect it to look like? More like AlexNet's `fc7`, or different?
4. Cichy et al. found that early network layers match early brain responses and later layers match later ones. Based on your nearest neighbours, what kind of information would you expect in the brain 100 ms vs. 200 ms after seeing a lion?